# Step 1: Choose plates to analyze [PLATE_DIRS] and output location for analysis files [OUTPUT_DIR]

In [ ]:
#Cell 1 — Imports & config
import os
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sys
import importlib
import utils.visualization
from utils.quiver_style import QUIVER_NEURON_COLORS
from utils.scs_utils import compute_psp_props_fov
from utils.data_utils import compute_neuron_snr, compute_ap_shape, mean_ap_shape
from utils.visualization import (
    build_condition_feature_df, filter_fovs,
    build_variance_table, _cohens_d,
    plot_condition_heatmap, plot_plate_heatmap,
    plot_well_feature_heatmap, plot_plate_condition_heatmap,
    plot_network_fingerprint, plot_circuit_composition,
    plot_fanin_fanout, plot_feature_boxplots, pool_conditions,
    plot_pharmacology_dissociation, plot_synaptic_coupling,
    plot_pharmacology_summary, plot_power_analysis, build_condition_colors,
    plot_condition_psp_waveforms, compute_all_psp_waveforms,
    build_ap_feature_df, plot_ap_heatmap, plot_connected_pair_heatmap,
    plot_ei_balance_debug,
)

# Choose folder to save this notebook's output
from datetime import date
OUTPUT_DIR = os.path.join(r'R:\QNP\2026_QNP\QNP066_069-PlateComparisons\Testing_DLX_protocols',
date.today().strftime('%Y-%m-%d'))
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Point each entry to a plate's OUTPUT_DIR
PLATE_DIRS = {
    #'QNP-066_Plate2': r'D:\Projects_Analysis\SCS\2026-05-08_JenniferGrooms_FireflyOne\PlateNumber_2\v17_outputs_20260529_1136', # No Drugs
    #'QNP-066_Plate3': r'D:\Projects_Analysis\SCS\2026-05-08_JenniferGrooms_FireflyOne_QNP-066\PlateNumber_3\v17_outputs_20260603_1536', # No Drugs
    #'QNP-068_Plate1': r'D:\Projects_Analysis\SCS\2026-05-12_JenniferGrooms_FireflyOne\PlateNumber_1\v17_outputs_20260608_1621', # GABAzine
    #'QNP-068_Plate2': r'D:\Projects_Analysis\SCS\2026-05-12_JenniferGrooms_FireflyOne\PlateNumber_2\v17_outputs_20260528_1652', # NBQX
    #'QNP-068_Plate3': r'D:\Projects_Analysis\SCS\2026-05-13_JenniferGrooms_FireflyOne\PlateNumber_3\v17_outputs_20260603_1552', # Drugs
    #'QNP-068_Plate4': r'D:\Projects_Analysis\SCS\2026-05-13_JenniferGrooms_FireflyOne\PlateNumber_4\v17_outputs_20260608_0848', # Drugs
    #'QNP-069_Plate1': r'D:\Projects_Analysis\SCS\2026-05-19_JenniferGrooms_FireflyOne\PlateNumber_1\v17_outputs_20260615_0949', # new DLX protocol
    'QNP-069_Plate2': r'D:\Projects_Analysis\SCS\2026-05-19_JenniferGrooms_FireflyOne\PlateNumber_2_forGrace\v17_outputs_20260618_1739', # new DLX protocol
    #'QNP-069_Plate3': r'D:\Projects_Analysis\SCS\2026-05-20_JenniferGrooms_FireflyOne\PlateNumber_3_forGrace\v17_outputs_20260601_1539', # new DLX protocol
    #'QNP-069_Plate4': r'D:\Projects_Analysis\SCS\2026-05-20_JenniferGrooms_FireflyOne\PlateNumber_4\v17_outputs_20260601_1551', # new DLX protocol

}

CONTROL_LABELS = {'No Drug', 'NoDrug', 'DMSO', 'dmso', ''}
THRESHOLD_MULT = 20                                        # Threshold for outlier status (20x above median)
exclude_outliers = False

# Step 2: Load previously analyzed data from pickle files. 
#### Approximately 2 minutes to load per plate

In [ ]:
# Cell 2 — Load data from each plate
all_fov_results = []
df_conn_list    = []
df_neuron_list  = []
df_dlx_list     = []

for plate_label, out_dir in PLATE_DIRS.items():
    # pickle
    pkl = os.path.join(out_dir, 'all_fov_results_cache.pkl')
    if os.path.exists(pkl):
        with open(pkl, 'rb') as f:
            plate_results = pickle.load(f)
        for R in plate_results:
            R['_plate'] = plate_label
        all_fov_results.extend(plate_results)
    else:
        print(f'[{plate_label}] no pickle found — skipping FOV-level analyses')

    # CSVs
    for fname, lst in (
        ('merged_connections_all_fovs.csv', df_conn_list),
        ('merged_neurons_all_fovs.csv',     df_neuron_list),
    ):
        path = os.path.join(out_dir, fname)
        if os.path.exists(path):
            df = pd.read_csv(path)
            df['plate'] = plate_label
            lst.append(df)
        else:
            print(f'[{plate_label}] {fname} not found')

    dlx_path = os.path.join(out_dir, 'dlx_accuracy_per_fov.csv')
    if os.path.exists(dlx_path):
        df = pd.read_csv(dlx_path)
        df['plate'] = plate_label
        df_dlx_list.append(df)
fov_well_map      = {}
fov_condition_map = {}
for plate_label, out_dir in PLATE_DIRS.items():
    path = os.path.join(out_dir,
'merged_connections_all_fovs_with_metadata.csv')
    if os.path.exists(path):
        available = pd.read_csv(path, nrows=0).columns.tolist()
        cols = [c for c in ('fov', 'wellId', 'drug1') if c in available]
        md = pd.read_csv(path, usecols=cols)
        for fov, well in md.groupby('fov')['wellId'].first().items():
            fov_well_map[(fov, plate_label)] = well
        if 'drug1' in md.columns:
            for fov, cond in md.groupby('fov')['drug1'].first().items():
                _cond_str = str(cond).strip() if pd.notna(cond) else ''
                fov_condition_map[(fov, plate_label)] = (
                    'No Drug' if not _cond_str or _cond_str.lower() in {'unknown', 'nan', 'none'} else _cond_str
                        )

df_conn    = pd.concat(df_conn_list,   ignore_index=True) if df_conn_list   else pd.DataFrame()
df_neurons = pd.concat(df_neuron_list, ignore_index=True) if df_neuron_list else pd.DataFrame()
df_dlx     = pd.concat(df_dlx_list,    ignore_index=True) if df_dlx_list    else pd.DataFrame()

print(f'Loaded {len(all_fov_results)} FOVs across {len(PLATE_DIRS)} plates')
print(f'  Connections: {len(df_conn)} rows')
print(f'  Neurons:     {len(df_neurons)} rows')
print(f'  DLX FOVs:    {len(df_dlx)} rows')

In [ ]:
# Find outlier FOVs with extreme connection counts or AP counts (exclude)
def flag_outlier_fovs(df_conn, all_fov_results, validated_col='scs_validated',
                        threshold_mult=10):
    import warnings

    ap_rows = []
    for R in all_fov_results:
        scs = R.get('scs1')
        if not scs:
            continue
        n_aps = sum(len(v) for v in scs.get('aps_all', {}).values())
        ap_rows.append({'fov': R['fov'], 'plate': R['_plate'], 'n_aps': n_aps})

    df_ap = pd.DataFrame(ap_rows) if ap_rows else pd.DataFrame(columns=['fov', 'plate', 'n_aps'])
    if not df_ap.empty:
        med_ap = df_ap['n_aps'].median()
        df_ap['flagged_ap'] = df_ap['n_aps'] > threshold_mult * med_ap

    conn_summary = (df_conn.groupby(['fov', 'plate'])
                    .agg(
                        n_connections=(validated_col, 'count'),
                        n_validated  =(validated_col, 'sum'),
                    )
                    .reset_index())

    summary = df_ap.merge(conn_summary, on=['fov', 'plate'], how='left')
    summary['n_connections'] = summary['n_connections'].fillna(0).astype(int)
    summary['n_validated']   = summary['n_validated'].fillna(0).astype(int)

    med_conn = summary['n_connections'].median()
    med_val  = summary['n_validated'].median()
    summary['flagged_conn'] = (
        (summary['n_connections'] > threshold_mult * med_conn) |
        (summary['n_validated']   > threshold_mult * med_val)
    )

    flagged_c = summary[summary['flagged_conn']]
    flagged_a = summary[summary['flagged_ap']]

    if not flagged_c.empty:
        warnings.warn(
            f"\n{'!'*60}\n"
            f"  OUTLIER FOVs — CONNECTION COUNT (>{threshold_mult}x median)\n" +
            flagged_c[['fov', 'plate', 'n_connections', 'n_validated']].to_string(index=False) +
            f"\n{'!'*60}", stacklevel=2
        )
    if not flagged_a.empty:
        warnings.warn(
            f"\n{'!'*60}\n"
            f"  OUTLIER FOVs — AP COUNT — WILL BE EXCLUDED (>{threshold_mult}x median)\n" +
            flagged_a[['fov', 'plate', 'n_aps']].to_string(index=False) +
            f"\n{'!'*60}", stacklevel=2
        )
    return summary

fov_qc = flag_outlier_fovs(df_conn, all_fov_results)
_flagged = fov_qc[fov_qc['flagged_ap'] | fov_qc['flagged_conn']]
if _flagged.empty:
    print("No outlier FOVs detected.")
else:
    display(_flagged.sort_values('n_aps', ascending=False))

ap_outlier_keys = set(zip(
    fov_qc.loc[fov_qc['flagged_ap'], 'fov'],
    fov_qc.loc[fov_qc['flagged_ap'], 'plate']
))
# Exclude connection outliers from df_conn
if exclude_outliers == True:
    flagged_conn_fovs = fov_qc.loc[fov_qc['flagged_conn'], ['fov', 'plate']]
    if not flagged_conn_fovs.empty:
        mask = df_conn.set_index(['fov', 'plate']).index.isin(
            flagged_conn_fovs.set_index(['fov', 'plate']).index)
        df_conn = df_conn[~mask].reset_index(drop=True)

    # Exclude AP outliers from df_conn and all_fov_results
    ap_outlier_keys = set(zip(
        fov_qc.loc[fov_qc['flagged_ap'], 'fov'],
        fov_qc.loc[fov_qc['flagged_ap'], 'plate']
    ))
    if ap_outlier_keys:
        mask = df_conn.set_index(['fov', 'plate']).index.isin(ap_outlier_keys)
        df_conn = df_conn[~mask].reset_index(drop=True)
        all_fov_results = [R for R in all_fov_results
                            if (R['fov'], R['_plate']) not in ap_outlier_keys]
        print(f"Excluded {len(ap_outlier_keys)} FOV(s) for AP outliers: {ap_outlier_keys}")

    # Exclude connection outliers from df_conn and all_fov_results
    conn_outlier_keys = set(zip(
        fov_qc.loc[fov_qc['flagged_conn'], 'fov'],
        fov_qc.loc[fov_qc['flagged_conn'], 'plate']
    ))
    if conn_outlier_keys:
        mask = df_conn.set_index(['fov', 'plate']).index.isin(
            flagged_conn_fovs.set_index(['fov', 'plate']).index)
        df_conn = df_conn[~mask].reset_index(drop=True)
        all_fov_results = [R for R in all_fov_results
                            if (R['fov'], R['_plate']) not in conn_outlier_keys]
        print(f"Excluded {len(conn_outlier_keys)} FOV(s) for connection outliers: {conn_outlier_keys}")

In [ ]:
############### TEMPORARY RELOAD BLOCK #######################################################
##############################################################################################
import importlib
import utils.scs_utils
import utils.visualization

importlib.reload(utils.scs_utils)
importlib.reload(utils.visualization)

from utils.visualization import (
    build_condition_feature_df, filter_fovs,
    build_variance_table, _cohens_d,
    plot_condition_heatmap, plot_plate_heatmap,
    plot_well_feature_heatmap, plot_plate_condition_heatmap,
    plot_network_fingerprint, plot_circuit_composition,
    plot_fanin_fanout, plot_feature_boxplots, pool_conditions,
    plot_pharmacology_dissociation, plot_synaptic_coupling,
    plot_pharmacology_summary, plot_power_analysis, build_condition_colors,
    plot_condition_psp_waveforms, compute_all_psp_waveforms,
    build_ap_feature_df, plot_ap_heatmap, plot_connected_pair_heatmap,
    plot_ei_balance_debug,
)
##############################################################################################
##############################################################################################

import matplotlib.pyplot as plt
plt.close('all')

In [ ]:
# Cell 3 =============== Build per-FOV feature table =============================================
df_features = build_condition_feature_df(all_fov_results)
df_features['plate'] = [R['_plate'] for R in all_fov_results]
df_features['well'] = [fov_well_map.get((R['fov'], R['_plate'])) for R in all_fov_results]
df_features['condition'] = [fov_condition_map.get((R['fov'], R['_plate']), 'No Drug') for R in all_fov_results]
# spike features
df_ap = build_ap_feature_df(all_fov_results)
df_ap['plate']     = [R['_plate'] for R in all_fov_results]
df_ap['well']      = [fov_well_map.get((R['fov'], R['_plate'])) for R in all_fov_results]
df_ap['condition'] = [fov_condition_map.get((R['fov'], R['_plate']), 'No Drug') for R in
all_fov_results]

cond_colors = build_condition_colors(df_features)
if not df_dlx.empty:
    keep = [c for c in ('fov', 'plate', 'pct_inh_correct', 'pct_exc_correct','threshold') if c in df_dlx.columns]
    df_features = df_features.merge(
        df_dlx[keep].rename(columns={
            'pct_inh_correct': 'dlx_pct_inh_correct',
            'pct_exc_correct': 'dlx_pct_exc_correct',
            'threshold':       'dlx_threshold',
        }),
        on=['fov', 'plate'], how='left',
    )


In [ ]:
# Cell 4 — Save run manifest
import json, platform, sys
from datetime import datetime

def _default(obj):
    if isinstance(obj, set):  return sorted(obj)
    if hasattr(obj, 'item'):  return obj.item()   # numpy scalars
    raise TypeError(type(obj))

manifest = {
    "run_timestamp": datetime.now().isoformat(timespec='seconds'),
    "output_dir":    OUTPUT_DIR,
    "python":        sys.version,
    "platform":      platform.platform(),

    "plates": {
        name: str(path) for name, path in PLATE_DIRS.items()
    },

    "control_labels": list(CONTROL_LABELS),

    "qc": {
        "exclude_outliers":  exclude_outliers,
        "threshold_mult":    THRESHOLD_MULT,
        "n_plates":          len(PLATE_DIRS),
        "n_fovs_total":      len(fov_qc),
        "n_fovs_flagged_ap": int(fov_qc['flagged_ap'].sum()),
        "n_fovs_flagged_conn": int(fov_qc['flagged_conn'].sum()),
    },
}

manifest_path = os.path.join(OUTPUT_DIR, 'run_manifest.json')
# Will also add excluded FOVs after this

# Exclude FOVs (optional)

In [ ]:
# # Cell 5 ======================== Exclude particular FOVs ==============================================

#df_filter = df_features[~((df_features['fov'] == 'FOV_0047') & (df_features['plate'] == 'QNP-068_Plate1'))]

# explicit exclusions
#df_filter = filter_fovs(df_features, exclude=['fov_0001'])

# keep only first 3 FOVs per well (useful if later acquisitions drift)
#df_filter = filter_fovs(df_features, max_per_well=3)

# combine — filters apply in order: exclude → max_fov → max_per_well
#df_filter = filter_fovs(df_features, max_fov=25, exclude=['fov_007'], max_per_well=4)

CONDITION_ORDER = ['No Drug', 'DAP5-NBQX-CNQX', 'GABAzine', 'Cyclothiazide', 'Forskolin', 'Ketamine']

df_filter = filter_fovs(df_features, max_fov_per_plate=36)
df_ap_filter = filter_fovs(df_ap, max_fov_per_plate=36)
cond_colors = build_condition_colors(df_filter, condition_order=CONDITION_ORDER)

_excl = df_features[
    ~df_features[['plate', 'fov']].apply(tuple, axis=1).isin(
        df_filter[['plate', 'fov']].apply(tuple, axis=1))
]
manifest['qc']['n_fovs_after_filter'] = len(df_filter['fov'].unique())
manifest['qc']['excluded_fovs'] = sorted(
    f"{r.plate}/{r.fov}" for _, r in _excl[['plate', 'fov']].iterrows()
)

manifest_path = os.path.join(OUTPUT_DIR, 'run_manifest.json')
with open(manifest_path, 'w') as f:
    json.dump(manifest, f, indent=2, default=_default)
print(f"Saved manifest: {manifest_path}")

# Generate visualizations

In [ ]:
X_SCALE = 0.3  # controls spacing between bars

In [ ]:
# Cell 6 ============ Connection counts (EXC / INH) and confidence scores ===========================
val_tiers = {'both', 'scs_only', 'sta_only'}
plates = list(PLATE_DIRS.keys())
if not df_conn.empty and 'consensus_tier' in df_conn.columns:
    val = df_conn[df_conn['consensus_tier'].isin(val_tiers)].copy()
    val['type'] = val['type'].str.upper()
    val['score'] = val[['scs_score_p1', 'sta_score_p1']].max(axis=1)

    # Remove outlier FOVs (connection count > Q3 + 3*IQR per plate/type)
    fov_counts = val.groupby(['plate', 'fov', 'type']).size().reset_index(name='n')
    thresholds = (fov_counts.groupby(['plate', 'type'])['n']
                .quantile([0.25, 0.75]).unstack()
                .rename(columns={0.25: 'q1', 0.75: 'q3'})
                .reset_index())
    thresholds['limit'] = thresholds['q3'] + 3 * (thresholds['q3'] - thresholds['q1'])
    fov_counts_clean = (fov_counts
                        .merge(thresholds[['plate', 'type', 'limit']], on=['plate', 'type'])
                        .query('n <= limit')
                        .drop(columns='limit'))
    clean_fovs = fov_counts_clean[['plate', 'fov']].drop_duplicates()
    val_clean = val.merge(clean_fovs, on=['plate', 'fov'])

    fig, axes = plt.subplots(1, 2, figsize=(max(6, len(plates) * 1.8), 3))

    # Connection counts
    counts = (val_clean.groupby(['plate', 'fov', 'type'])
            .size().reset_index(name='n')
            .groupby(['plate', 'type'])['n'].agg(['mean', 'sem'])
            .reset_index())
    
    x = np.arange(len(plates))
    width = 0.3

    for j, typ in enumerate(['EXC', 'INH']):
        color = QUIVER_NEURON_COLORS['Inhibitory'] if typ == 'INH' else QUIVER_NEURON_COLORS['Excitatory']
        sub = counts[counts['type'] == typ].set_index('plate').reindex(plates)
        offset = (j - 0.5) * width
        axes[0].bar(x + offset, sub['mean'], yerr=sub['sem'],
                    width=width, color=color, capsize=4, label=typ)

    axes[0].set_xticks(x)
    axes[0].set_xticklabels(plates, ha='center')
    axes[0].set_ylabel('Mean connections per FOV')
    axes[0].set_title('EXC / INH connection counts')
    axes[0].legend(frameon=False)
    axes[0].spines[['top', 'right']].set_visible(False)

    # Confidence scores
    for i, (plate, grp) in enumerate(val_clean.groupby('plate')):
        for j, (typ, color) in enumerate([('EXC', QUIVER_NEURON_COLORS['Excitatory']), ('INH', QUIVER_NEURON_COLORS['Inhibitory'])]):
            scores = grp.loc[grp['type'] == typ, 'score'].dropna()
            axes[1].scatter([i + j * 0.3] * len(scores), scores,
                            color=color, s=10, alpha=0.4)
            if len(scores):
                axes[1].plot([i + j * 0.3], [scores.mean()],
                            'o', color=color, markersize=7, zorder=5)

    axes[1].set_xticks(range(len(PLATE_DIRS)))
    axes[1].set_xticks([i + 0.15 for i in range(len(PLATE_DIRS))])
    axes[1].set_xticklabels(list(PLATE_DIRS.keys()), ha='center', fontsize=8)
    axes[1].set_ylabel('Confidence score')
    axes[1].set_title('Connection confidence scores')
    axes[1].spines[['top', 'right']].set_visible(False)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'connection_counts_confidence.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
#Cell 7 ======================= PSP properties per plate ============================================
if not df_conn.empty:
    psp_rows = []
    for R in all_fov_results:
        props = compute_psp_props_fov(R)
        if props is None:
            continue
        row = {'fov': R['fov'], 'plate': R['_plate']}
        for typ in ('EXC', 'INH'):
            for k, v in props[typ].items():
                row[f'{k}_{typ.lower()}'] = v
        psp_rows.append(row)
    df_psp = pd.DataFrame(psp_rows)

    _psp_props = [
        ('amplitude',      'Amplitude (ΔF/F)'),
        ('auc',            'AUC (ΔF/F · s)'),
        ('rise_time_ms',   'Rise time (ms)'),
        ('half_width_ms',  'Half-width (ms)'),
        ('decay_time_ms',  'Decay time to 50% (ms)'),
        ('snr',            'SNR'),
    ]

    plates = list(PLATE_DIRS.keys())
    fig, axes = plt.subplots(2, 3, figsize=(8, 8))
    for ax_idx, (ax, (prop, ylabel)) in enumerate(zip(axes.flatten(), _psp_props)):
        for i, plate in enumerate(plates):
            grp = df_psp[df_psp['plate'] == plate]
            for j, (typ, color) in enumerate([('exc', QUIVER_NEURON_COLORS['Excitatory']), ('inh', QUIVER_NEURON_COLORS['Inhibitory'])]):
                vals = grp[f'{prop}_{typ}'].dropna()
                if ax_idx == 5:
                    vals = vals[vals <= 1000]
                n = len(vals)
                mean = vals.mean() if n else 0
                sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
                x = i * 2.5 + j * 0.9
                ax.bar(x, mean, yerr=sem, color=color, width=0.7, capsize=4,
                        error_kw=dict(elinewidth=1.2),
                        label=typ.upper() if i == 0 else '')
                ax.scatter([x] * n, vals, color='k', s=12, zorder=5, alpha=0.5)
        ax.set_xticks([i * 2.5 + 0.45 for i in range(len(plates))])
        ax.set_xticklabels(plates, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel(ylabel)
        ax.spines[['top', 'right']].set_visible(False)
        if ax is axes.flatten()[0]:
            ax.legend(frameon=False, fontsize=8)
    plt.suptitle('PSP properties across plates', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'psp_properties.png'), dpi=300,
    bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 8 ========================= DLX accuracy per plate ===========================================
if not df_dlx.empty:
    fig, axes = plt.subplots(1, 2, figsize=(6, 4))
    for ax, (col, label, color) in zip(axes, [
        ('pct_inh_correct', 'INH correctly DLX+', QUIVER_NEURON_COLORS['Inhibitory']),
        ('pct_exc_correct', 'EXC correctly DLX−', QUIVER_NEURON_COLORS['Excitatory']),
    ]):
        for i, (plate, grp) in enumerate(df_dlx.groupby('plate')):
            vals = grp[col].dropna()
            n = len(vals)
            mean = vals.mean() if n else 0
            sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
            ax.bar(i, mean, yerr=sem, color=color, width=0.6, capsize=5,
                    error_kw=dict(elinewidth=1.5))
            ax.scatter([i] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
            ax.text(i, 2, f'n={n}', ha='center', va='bottom', fontsize=8)
        ax.set_xticks(range(df_dlx['plate'].nunique()))
        ax.set_xticklabels(df_dlx['plate'].unique(), rotation=30, ha='right')
        ax.set_ylabel(f'{label} (%)')
        ax.set_ylim(0, 110)
        ax.set_title(label)
        ax.spines[['top', 'right']].set_visible(False)
    plt.suptitle('DLX cell identity accuracy', fontsize=12)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'neuron_composition_DLX.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
#  Cell 9 ======== Validation rate per FOV — EXC / INH strip plot across plates ====================
if not df_conn.empty:
    total = (df_conn.groupby(['plate', 'fov', 'type'])
            .size().reset_index(name='total'))
    validated = (df_conn[df_conn['consensus_tier'].isin(val_tiers)]
                .groupby(['plate', 'fov', 'type'])
                .size().reset_index(name='n_val'))
    vr = total.merge(validated, on=['plate', 'fov', 'type'], how='left')
    vr['n_val'] = vr['n_val'].fillna(0)
    vr['rate'] = 100 * vr['n_val'] / vr['total']
    vr['type'] = vr['type'].str.upper()

    plates = list(PLATE_DIRS.keys())
    fig, ax = plt.subplots(figsize=(max(3, len(plates) * 1.2), 3))

    for i, plate in enumerate(plates):
        for j, (typ, color) in enumerate([('EXC', QUIVER_NEURON_COLORS['Excitatory']), ('INH', QUIVER_NEURON_COLORS['Inhibitory'])]):
            rates = vr.loc[(vr['plate'] == plate) & (vr['type'] == typ), 'rate'].dropna()
            x = i * 1.2 + j * 0.5
            x_jit = np.random.uniform(x - 0.2, x + 0.2, size=len(rates))
            ax.scatter(x_jit, rates, color=color, s=18, alpha=0.5,
                        label=typ if i == 0 else '')
            if len(rates):
                ax.plot(x, rates.mean(), 'o', color=color, markersize=8,
                        markeredgecolor='k', markeredgewidth=0.8, zorder=5)

    ax.set_xticks([i * 1.2 + 0.25 for i in range(len(plates))])
    ax.set_xticklabels(plates, rotation=0, ha='center', fontsize=8)
    ax.set_ylabel('Validation rate (%)')
    ax.set_title('Validation rate per FOV across plates')
    #ax.legend(frameon=False)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'validation_rate_strip.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
#  Cell 10 ======================== Pharmacological validation =====================================
if not df_conn.empty:
    GABAZINE = 'GABAzine'
    CNQX     = 'DAP5-NBQX-CNQX'  # adjust to match your condition label
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    # Build per-FOV metrics for all four conditions
    rows = []
    for R in all_fov_results:
        fov   = R['fov']
        cond  = str(fov_condition_map.get((fov, R['_plate']), '') or '').strip() or 'No Drug'
        if cond not in ('No Drug', GABAZINE, CNQX):
            continue

        scs    = R.get('scs1')
        merged = R.get('merged')
        meta   = R.get('meta', {})
        fs     = int(meta.get('fs', 500) or 500)
        rec_s  = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs

        epsp_n = sum(len(v) for v in scs.get('epsps_all', {}).values()) if scs else 0
        ipsp_n = sum(len(v) for v in scs.get('ipsps_all', {}).values()) if scs else 0

        if merged is not None and not merged.empty:
            inh = merged[merged['type'].str.upper() == 'INH']
            exc = merged[merged['type'].str.upper() == 'EXC']
            inh_val = inh['consensus_tier'].isin(VAL_TIERS).sum()
            exc_val = exc['consensus_tier'].isin(VAL_TIERS).sum()
            inh_scores = pd.concat([inh[c].dropna() for c in
    ('scs_score_p1','sta_score_p1') if c in inh.columns])
            exc_scores = pd.concat([exc[c].dropna() for c in
    ('scs_score_p1','sta_score_p1') if c in exc.columns])
            inh_score = float(inh_scores.mean()) if len(inh_scores) else np.nan
            exc_score = float(exc_scores.mean()) if len(exc_scores) else np.nan
        else:
            inh_val = exc_val = 0
            inh_score = exc_score = np.nan

        rows.append({'condition': cond,
                    'ipsp_rate': ipsp_n / rec_s if rec_s else np.nan,
                    'epsp_rate': epsp_n / rec_s if rec_s else np.nan,
                    'inh_val':   inh_val,
                    'exc_val':   exc_val,
                    'inh_score': inh_score,
                    'exc_score': exc_score})

    df_pharm = pd.DataFrame(rows)

    _pharm_drugs = {GABAZINE, CNQX} & set(df_pharm['condition'].unique())
    if not _pharm_drugs:
        print("  No GABAzine or CNQX conditions found — pharmacological validation skipped.")
    else:

        # Plot
        fig, axes = plt.subplots(2, 3, figsize=(8, 6))

        panels = [
            # (ax_row, ax_col, metric,      No Drug col,   Drug col,   drug, color,     ylabel)
            (0, 0, 'ipsp_rate',  'No Drug', GABAZINE, '#DD52D1', 'IPSP rate (Hz)'),
            (0, 1, 'inh_val',    'No Drug', GABAZINE, '#DD52D1', 'INH validated connections'),
            (0, 2, 'inh_score',  'No Drug', GABAZINE, '#DD52D1', 'INH confidence score'),
            (1, 0, 'epsp_rate',  'No Drug', CNQX,     "#FD0B70", 'EPSP rate (Hz)'),
            (1, 1, 'exc_val',    'No Drug', CNQX,     '#FD0B70', 'EXC validated connections'),
            (1, 2, 'exc_score',  'No Drug', CNQX,     '#FD0B70', 'EXC confidence score'),
        ]

        for row, col, metric, ctrl_lbl, drug_lbl, drug_color, ylabel in panels:
            ax = axes[row, col]
            conditions = [ctrl_lbl, drug_lbl]
            colors     = ['gray', drug_color]

            ctrl_vals = df_pharm.loc[df_pharm['condition'] == ctrl_lbl,
        metric].dropna()
            drug_vals = df_pharm.loc[df_pharm['condition'] == drug_lbl,
        metric].dropna()
            d = _cohens_d(drug_vals.values, ctrl_vals.values)

            for i, (cond, color) in enumerate(zip([ctrl_lbl, drug_lbl], ['gray', drug_color])):
                x    = i * X_SCALE
                vals = df_pharm.loc[df_pharm['condition'] == cond, metric].dropna()
                n    = len(vals)
                mean = vals.mean() if n else 0
                sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
                ax.bar(x, mean, yerr=sem, color=color, width=0.3,
                        capsize=5, error_kw=dict(elinewidth=1.5))
                ax.scatter([x] * n, vals, color='k', s=15, zorder=5, alpha=0.6)
                #ax.text(x, 0, f'n={n} FOV', ha='center', va='bottom', fontsize=6)

            ax.set_xticks([i * X_SCALE for i in range(2)])
            ax.set_xticklabels([ctrl_lbl, drug_lbl], rotation=20, ha='right', fontsize=7)
            ax.set_ylabel(ylabel, fontsize=10)
            ax.set_title(f'cohen d={d:.2f}' if not np.isnan(d) else '', fontsize=8)
            ax.spines[['top', 'right']].set_visible(False)
            ax.set_title(f'cohen d={d:.2f}' if not np.isnan(d) else '', fontsize=8)
            if col == 1:
                if row == 1:
                    ax.set_ylim(bottom=0, top=500)
                else:
                    ax.set_ylim(bottom=0, top=150)
            ax.spines[['top', 'right']].set_visible(False)

        fig.text(0, 0.75, 'GABAzine', va='center', rotation='vertical',
                fontsize=8, fontweight='bold', color='#DD52D1')
        fig.text(0, 0.25, 'CNQX', va='center', rotation='vertical',
                fontsize=8, fontweight='bold', color='#FD0B70')

        plt.suptitle('Pharmacological validation — INH and EXC blockade', fontsize=11)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'pharmacological_validation.png'),
        dpi=300, bbox_inches='tight')
        plt.close()

In [ ]:
# Cell 11 ================== Scaling within the identified network ================================
# Stim APs vs evoked partner responses, normalized to chances 
# Shows the increase in evoked activity detection with AP rate (left)
# and probability of detecting a response per AP (right)
if not df_conn.empty:
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    neuron_rows = []
    for R in all_fov_results:
        merged = R.get('merged')
        sta    = R.get('sta1', {})
        if merged is None or merged.empty or not sta:
            continue

        val = merged[merged['consensus_tier'].isin(VAL_TIERS)]
        if val.empty:
            continue

        sta_results = sta.get('all_results', {})
        aps_for_sta = sta.get('aps_for_sta', {})

        for pre, grp in val.groupby('pre'):
            n_stim_aps = len(aps_for_sta.get(pre, []))
            if n_stim_aps == 0:
                continue

            corrected_total  = 0
            fractions        = []
            n_valid_partners = 0

            for _, conn in grp.iterrows():
                result = sta_results.get((pre, conn['post']))
                if result is None:
                    continue
                n_psp       = getattr(result, 'n_psp',           0) or 0
                n_ap_evoked = getattr(result, 'n_ap_evoked',     0) or 0
                n_coinc     = getattr(result, 'n_coinc_dropped', 0) or 0
                observable  = n_stim_aps - n_coinc
                if observable <= 0:
                    continue
                corrected_total += (n_psp + n_ap_evoked) * n_stim_aps / observable
                fractions.append((n_psp + n_ap_evoked) / observable)
                n_valid_partners += 1

            if n_valid_partners == 0 or corrected_total == 0:
                continue

            neuron_rows.append({
                'plate':             R['_plate'],
                'n_stim_aps':        n_stim_aps,
                'evoked_per_partner': corrected_total / n_valid_partners,
                'response_fraction': np.mean(fractions),
            })

    df_neuron = pd.DataFrame(neuron_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = (_iqr_mask(df_neuron['n_stim_aps']) &
            _iqr_mask(df_neuron['evoked_per_partner']) &
            _iqr_mask(df_neuron['response_fraction']))
    df_p = df_neuron[mask]
    n_removed = len(df_neuron) - mask.sum()

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    for ax, ycol, ylabel, ylim in [
        (axes[0], 'evoked_per_partner', 'Evoked responses per partner\n(coincidence-corrected)',  None),
        (axes[1], 'response_fraction',  'Mean response fraction per partner\n(evoked / observable windows)', (0, 1)),
    ]:
        x = df_p['n_stim_aps'].values
        y = df_p[ycol].values

        ax.scatter(x, y, color='#5B8DB8', s=15, alpha=0.5)

        if len(x) > 2:
            slope, intercept, r, p, _ = stats.linregress(x, y)
            xfit = np.linspace(x.min(), x.max(), 100)
            ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
            ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                    transform=ax.transAxes, fontsize=8)

        ax.set_xlabel('Pre-synaptic APs during stimulation', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f'n={len(df_p)} neurons, {n_removed} removed', fontsize=8)
        ax.set_xlim(left=0)
        ax.set_ylim(bottom=0) if ylim is None else ax.set_ylim(ylim)
        ax.spines[['top', 'right']].set_visible(False)

    axes[0].set_title(f'Count: scales with firing\n(n={len(df_p)} neurons, {n_removed} removed)', fontsize=10)
    axes[1].set_title(f'Fraction: per-spike reliability\n(n={len(df_p)} neurons, {n_removed} removed)', fontsize=10)

    fig.suptitle('Stim APs vs evoked partner responses', fontsize=10)
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR,
    'stim_aps_vs_evoked_count_and_fraction.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 12 =============== EXC pre AP vs post EPSP rate per FOV ===================================
if not df_conn.empty:
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    fov_rows = []
    for R in all_fov_results:
        scs        = R.get('scs1')
        merged     = R.get('merged')
        meta       = R.get('meta', {})
        sta_results = R.get('sta1', {}).get('all_results', {})
        if not scs or merged is None or merged.empty:
            continue

        fs    = int(meta.get('fs', 500) or 500)
        rec_s = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs
        if rec_s == 0:
            continue

        val_exc = merged[
            merged['consensus_tier'].isin(VAL_TIERS) &
            (merged['type'].str.upper() == 'EXC')
        ]
        if val_exc.empty:
            continue

        pre_neurons  = set(val_exc['pre'])
        post_neurons = set(val_exc['post'])

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})

        # build EPSP count per post neuron, filling sta_only gaps from STA n_psp
        epsp_counts = {k: len(v) for k, v in epsps_all.items()}
        for _, conn in val_exc[val_exc['consensus_tier'] ==
    'sta_only'].iterrows():
            pre, post = conn['pre'], conn['post']
            if epsp_counts.get(post, 0) == 0:
                result = sta_results.get((pre, post))
                if result is not None and hasattr(result, 'n_psp'):
                    epsp_counts[post] = epsp_counts.get(post, 0) + result.n_psp

        pre_ap_rates    = [len(v) / rec_s for k, v in aps_all.items()
                            if k in pre_neurons and len(v) > 0]
        post_epsp_rates = [epsp_counts.get(k, 0) / rec_s for k in post_neurons
                            if epsp_counts.get(k, 0) > 0]

        if not pre_ap_rates or not post_epsp_rates:
            continue

        fov_rows.append({
            'plate':     R['_plate'],
            'ap_rate':   np.mean(pre_ap_rates),
            'epsp_rate': np.mean(post_epsp_rates),
        })

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['ap_rate']) & _iqr_mask(df_fov['epsp_rate'])
    df_filter = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_filter['ap_rate'].values
    y = df_filter['epsp_rate'].values

    ax.scatter(x, y, color=QUIVER_NEURON_COLORS['Excitatory'], s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Mean AP rate — EXC pre-synaptic neurons (Hz)', fontsize=8)
    ax.set_ylabel('Mean EPSP rate — EXC post-synaptic neurons (Hz)', fontsize=8)
    ax.set_title(f'EXC pre AP vs post EPSP rate per FOV\n(n={len(df_filter)} FOVs,{n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'exc_pre_ap_vs_post_epsp_rate.png'),
    dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 13 ============================ AP vs PSP rate — SNR≥ ======================================
if not df_conn.empty:
    SNR_GATE = 3.5

    fov_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        meta = R.get('meta', {})
        if not scs:
            continue

        n_sources = int(meta.get('n_sources', 0) or 0)
        if n_sources == 0:
            continue

        fs    = int(meta.get('fs', 500) or 500)
        rec_s = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs
        if rec_s == 0:
            continue

        aps_all    = scs.get('aps_all',    {})
        epsps_all  = scs.get('epsps_all',  {})
        ipsps_all  = scs.get('ipsps_all',  {})
        traces_all = scs.get('traces_all', {})

        mean_snr, above_gate = compute_neuron_snr(traces_all, aps_all,
    snr_gate=SNR_GATE)
        good_sources = {k for k, ok in above_gate.items() if ok}

        active_n = sum(1 for k, v in aps_all.items() if k in good_sources and
    len(v) > 0)
        if active_n == 0:
            continue

        n_aps  = sum(len(v) for k, v in aps_all.items()   if k in good_sources)
        n_psps = sum(len(v) for k, v in epsps_all.items() if k in good_sources) \
                + sum(len(v) for k, v in ipsps_all.items() if k in good_sources)

        n_good = len(good_sources)

        fov_rows.append({
            'plate':    R['_plate'],
            'ap_rate':  (n_aps  / active_n) / rec_s,
            'psp_rate': (n_psps / n_good)   / rec_s,
        })

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['ap_rate']) & _iqr_mask(df_fov['psp_rate'])
    df_filter = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_filter['ap_rate'].values
    y = df_filter['psp_rate'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Mean AP rate per active neuron (Hz)', fontsize=8)
    ax.set_ylabel('Mean PSP rate per high-SNR source (Hz)', fontsize=8)
    ax.set_title(f'AP vs PSP rate — SNR≥{SNR_GATE} sources\n(n={len(df_filter)} FOVs, {n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ap_vs_psp_rate_highsnr.png'), dpi=300,
    bbox_inches='tight')
    plt.close()


In [ ]:
# Cell 14 ========================== AP vs PSP counts per well ===================================
# # Total APs vs PSPs per well
if not df_conn.empty:
    well_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        if not scs:
            continue

        fov   = R['fov']
        plate = R['_plate']

        if (fov, plate) in ap_outlier_keys:
            continue
        well  = fov_well_map.get((fov, plate), 'unknown')
     
        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        n_aps  = sum(len(v) for v in aps_all.values())
        n_psps = sum(len(v) for v in epsps_all.values()) + sum(len(v) for v in ipsps_all.values())

        well_rows.append({'plate': plate, 'well': well, 'n_aps': n_aps, 'n_psps': n_psps})

    df_well = (pd.DataFrame(well_rows)
                .groupby(['plate', 'well'])[['n_aps', 'n_psps']]
                .sum()
                .reset_index())

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_well['n_aps'].values
    y = df_well['n_psps'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Total APs per well', fontsize=8)
    ax.set_ylabel('Total PSPs per well', fontsize=8)
    ax.set_title(f'AP vs PSP counts per well (n={len(df_well)} wells)',fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'ap_vs_psp_per_well.png'), dpi=300,
    bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 15 ======================== Sources vs total PSP count =====================================
# # Number of sources per FOV
if not df_conn.empty:
    fov_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        meta = R.get('meta', {})
        if not scs:
            continue

        n_sources = int(meta.get('n_sources', 0) or 0)
        if n_sources == 0:
            continue

        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        n_psps = sum(len(v) for v in epsps_all.values()) + sum(len(v) for v in
    ipsps_all.values())

        fov_rows.append({'plate': R['_plate'], 'n_sources': n_sources, 'n_psps':
    n_psps})

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['n_sources']) & _iqr_mask(df_fov['n_psps'])
    df_filter_sc = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_filter_sc['n_sources'].values
    y = df_filter_sc['n_psps'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Number of sources per FOV', fontsize=8)
    ax.set_ylabel('Total PSPs per FOV', fontsize=8)
    ax.set_title(f'Sources vs total PSP count\n(n={len(df_filter_sc)} FOVs, {n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'sources_vs_total_psps.png'), dpi=300,
    bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 16 ================ Stimulated neuron validated pairs only ================================
if not df_conn.empty:
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    neuron_rows = []
    for R in all_fov_results:
        merged      = R.get('merged')
        sta         = R.get('sta1', {})
        if merged is None or merged.empty or not sta:
            continue

        val = merged[merged['consensus_tier'].isin(VAL_TIERS)]
        if val.empty:
            continue

        sta_results  = sta.get('all_results', {})
        aps_for_sta  = sta.get('aps_for_sta', {})

        # group validated connections by pre neuron
        for pre, grp in val.groupby('pre'):
            n_stim_aps = len(aps_for_sta.get(pre, []))
            if n_stim_aps == 0:
                continue

            n_psp_total      = 0
            n_ap_evoked_total = 0
            for _, conn in grp.iterrows():
                post   = conn['post']
                result = sta_results.get((pre, post))
                if result is None:
                    continue
                n_psp_total       += getattr(result, 'n_psp',       0) or 0
                n_ap_evoked_total += getattr(result, 'n_ap_evoked', 0) or 0

            if n_psp_total == 0 and n_ap_evoked_total == 0:
                continue

            neuron_rows.append({
                'plate':          R['_plate'],
                'n_stim_aps':     n_stim_aps,
                'n_psp':          n_psp_total,
                'n_ap_evoked':    n_ap_evoked_total,
                'n_partners':     len(grp),
            })

    df_neuron = pd.DataFrame(neuron_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))

    for ax, ycol, ylabel, color in [
        (axes[0], 'n_psp',       'Total subthreshold PSPs in partners', "#043E70"),
        (axes[1], 'n_ap_evoked', 'Total evoked APs in partners', "#07030000"),
    ]:
        mask = _iqr_mask(df_neuron['n_stim_aps']) & _iqr_mask(df_neuron[ycol])
        df_p = df_neuron[mask]
        n_removed = len(df_neuron) - mask.sum()

        x = df_p['n_stim_aps'].values
        y = df_p[ycol].values
        ax.scatter(x, y, color=color, s=15, alpha=0.5)

        if len(x) > 2:
            slope, intercept, r, p, _ = stats.linregress(x, y)
            xfit = np.linspace(x.min(), x.max(), 100)
            ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
            ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                    transform=ax.transAxes, fontsize=8)

        ax.set_xlabel('APs fired during stimulation (pre neuron)', fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.set_title(f'n={len(df_p)} neurons, {n_removed} removed', fontsize=8)
        ax.spines[['top', 'right']].set_visible(False)

    fig.suptitle('Stimulation-evoked pre APs vs post-synaptic responses',
    fontsize=10)
    fig.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'stim_aps_vs_partner_responses.png'),
    dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 17 === For each FOV: active neurons (those with at least one AP) and total PSPs (EPSPs + IPSPs)
# removes extreme outliers using a 3×IQR filter, then plots the two against each other with a linear regression
if not df_conn.empty:
    fov_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        if not scs:
            continue

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        n_active = sum(1 for v in aps_all.values() if len(v) > 0)
        if n_active == 0:
            continue

        n_psps = sum(len(v) for v in epsps_all.values()) + sum(len(v) for v in
    ipsps_all.values())

        fov_rows.append({'plate': R['_plate'], 'n_active': n_active, 'n_psps':
    n_psps})

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['n_active']) & _iqr_mask(df_fov['n_psps'])
    df_filter_sc = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_filter_sc['n_active'].values
    y = df_filter_sc['n_psps'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Number of active neurons per FOV', fontsize=8)
    ax.set_ylabel('Total PSPs per FOV', fontsize=8)
    ax.set_title(f'Active neurons vs total PSP count\n(n={len(df_filter_sc)} FOVs,{n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'active_neurons_vs_total_psps.png'),
    dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 18 — 
if not df_conn.empty:
    fov_rows = []
    for R in all_fov_results:
        scs  = R.get('scs1')
        if not scs:
            continue

        aps_all   = scs.get('aps_all',   {})
        epsps_all = scs.get('epsps_all', {})
        ipsps_all = scs.get('ipsps_all', {})

        active = {k for k, v in aps_all.items() if len(v) > 0}
        if not active:
            continue

        n_active = len(active)
        n_aps    = sum(len(aps_all[k]) for k in active)
        n_psps   = sum(len(v) for v in epsps_all.values()) + sum(len(v) for v in
    ipsps_all.values())

        fov_rows.append({'plate': R['_plate'], 'n_aps': n_aps, 'n_psps': n_psps /
    n_active})

    df_fov = pd.DataFrame(fov_rows)

    def _iqr_mask(s, k=3.0):
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        return (s >= q1 - k * iqr) & (s <= q3 + k * iqr)

    mask = _iqr_mask(df_fov['n_aps']) & _iqr_mask(df_fov['n_psps'])
    df_filter_sc = df_fov[mask]
    n_removed = len(df_fov) - mask.sum()

    fig, ax = plt.subplots(figsize=(4, 3.5))
    x = df_filter_sc['n_aps'].values
    y = df_filter_sc['n_psps'].values

    ax.scatter(x, y, color='#5B8DB8', s=20, alpha=0.6)

    if len(x) > 2:
        slope, intercept, r, p, _ = stats.linregress(x, y)
        xfit = np.linspace(x.min(), x.max(), 100)
        ax.plot(xfit, slope * xfit + intercept, color='k', lw=1.5)
        ax.text(0.05, 0.92, f'R²={r**2:.2f}  p={p:.2e}',
                transform=ax.transAxes, fontsize=8)

    ax.set_xlabel('Total APs in active neurons', fontsize=8)
    ax.set_ylabel('Total PSPs per active neuron', fontsize=8)
    ax.set_title(f'APs vs PSPs — all detected PSPs\n(n={len(df_filter_sc)} FOVs, {n_removed} removed)', fontsize=9)
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'aps_vs_psps_all_detected.png'), dpi=300,
    bbox_inches='tight')
    plt.close()

In [ ]:
# Cell 19 ============ HEATMAP, NETWORK FINGERPRINT, CIRCUIT COMPOSITION, FAN PLOTS ============
df_filter = filter_fovs(df_features, max_fov_per_plate=36)
cond_colors = build_condition_colors(df_filter)

plot_condition_heatmap(df_filter, vmax=3, condition_order=CONDITION_ORDER,
                         output_path=os.path.join(OUTPUT_DIR,'plate_condition_heatmap.png'))
plot_connected_pair_heatmap(df_filter, df_ap_filter,
                              vmax=3, condition_order=CONDITION_ORDER,
                              output_path=os.path.join(OUTPUT_DIR, 'connected_pair_heatmap.png'))
plot_ap_heatmap(df_ap_filter,condition_order=CONDITION_ORDER,
                        output_path=os.path.join(OUTPUT_DIR, 'ap_heatmap.png'),)
plot_network_fingerprint(df_filter, condition_colors=cond_colors,
                        output_path=os.path.join(OUTPUT_DIR,'network_fingerprint.png'))
plot_circuit_composition(df_filter, condition_order=CONDITION_ORDER,
                        output_path=os.path.join(OUTPUT_DIR, 'circuit_composition.png'))
plot_fanin_fanout(df_filter, condition_colors=cond_colors, condition_order=CONDITION_ORDER,
                        output_path=os.path.join(OUTPUT_DIR, 'fanin_fanout.png'))

In [ ]:
# Cell 20 — 
plot_plate_heatmap(df_features, output_path=os.path.join(OUTPUT_DIR, 'plate_heatmap.png'))
plot_plate_condition_heatmap(df_features, vmax=4, output_path=os.path.join(OUTPUT_DIR,'plate_condition_heatmap.png'))
plot_well_feature_heatmap(df_features, cmap='Purples', output_path=os.path.join(OUTPUT_DIR,'well_feature_heatmap.png'))

In [ ]:
# Prints all the fovs in order for selecting individual fov indices
for i, R in enumerate(all_fov_results):
    fov   = R.get('fov')
    plate = R.get('_plate')
    cond  = fov_condition_map.get((fov, plate), 'unknown')
    print(f"{i:3d}  {cond:<25}  {fov}  {plate}")

In [ ]:
 # Pick one representative FOV per condition
# Find one FOV per condition
def pick_fov_per_condition(all_fov_results, fov_condition_map):
    seen = {}
    for R in all_fov_results:
        fov   = R.get('fov')
        plate = R.get('_plate')
        cond  = fov_condition_map.get((fov, plate))
        if cond:
            if cond not in seen:
                seen[cond] = [R]
            elif len(seen[cond]) < 2:
                seen[cond].append(R)
    return {cond: Rs[-1] for cond, Rs in seen.items()}

condition_fov_map = pick_fov_per_condition(all_fov_results, fov_condition_map)

condition_fov_map = pick_fov_per_condition(all_fov_results, fov_condition_map)
#condition_fov_map['No Drug'] = all_fov_results[1]  # optional override 
#condition_fov_map['Ketamine'] = all_fov_results[156] # optional override 

assert 'No Drug' in condition_fov_map, (f"'No Drug' condition not found. Available conditions: {list(condition_fov_map.keys())}")

plot_condition_psp_waveforms(condition_fov_map, output_path=os.path.join(OUTPUT_DIR, 'condition_psp_waveforms.png'))

In [ ]:
plot_ei_balance_debug(df_features, condition_order=CONDITION_ORDER,condition_colors=cond_colors,
                      output_path=os.path.join(OUTPUT_DIR, 'EI_summary.png'))

In [ ]:
# in progress — Pharmacology summary figure
# all_fov_results variable is optional: add in #all_fov_results=all_fov_results but beware it is slow
# all_fov_results gives a mean PSP trace for the entire plate
plot_pharmacology_summary(
    df_features, condition_colors=cond_colors, condition_order=CONDITION_ORDER,
    output_path=os.path.join(OUTPUT_DIR, 'pharmacology_summary.png')
)

In [ ]:
# in progress - check function
plot_pharmacology_dissociation(df_features, condition_colors=cond_colors, condition_order=CONDITION_ORDER,
                               output_path=os.path.join(OUTPUT_DIR, 'pharmacology_dissociation.png'))
plot_synaptic_coupling(df_features,condition_colors=cond_colors, condition_order=CONDITION_ORDER,
                       output_path=os.path.join(OUTPUT_DIR, 'synaptic_coupling.png'))

In [ ]:
# Cell XX — Boxplot of PSP and network features
only_no_drug = set(df_features['condition'].unique()) == {'No Drug'}

plot_feature_boxplots(df_features, condition_colors=cond_colors, condition_order=CONDITION_ORDER,
                      fold_change=not only_no_drug, 
                      output_path=os.path.join(OUTPUT_DIR, 'boxplots.png'))

In [ ]:
# Cell XX — Validation rate per plate
if not df_conn.empty:
    val_tiers = {'both', 'scs_only', 'sta_only'}
    df_conn['validated'] = df_conn['consensus_tier'].isin(val_tiers)

    val_rate = (df_conn.groupby(['plate', 'fov'])
                .agg(n_total=('validated', 'size'),
                    n_validated=('validated', 'sum'))
                .assign(pct_validated=lambda x: 100 * x.n_validated / x.n_total)
                .reset_index())
    width = 0.2
    fig, ax = plt.subplots(figsize=(len(PLATE_DIRS) * X_SCALE + 1.5, 3))
    for i, (plate, grp) in enumerate(val_rate.groupby('plate')):
        vals = grp['pct_validated']
        n = len(vals)
        mean, sem = vals.mean(), vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
        x = i * X_SCALE
        ax.bar(x, mean, yerr=sem, color='gray', width=width, capsize=5, error_kw=dict(elinewidth=1.5))
        ax.scatter([x] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
        ax.text(x, 2, f'{n} FOVs', ha='center', va='bottom', fontsize=6)

    ax.set_xticks([i * X_SCALE for i in range(len(PLATE_DIRS))])
    ax.set_xticklabels(list(PLATE_DIRS.keys()), rotation=30, ha='right', fontsize=7)
    ax.set_ylabel('Validated connections (%)')
    ax.set_title('Validation rate per plate')
    ax.spines[['top', 'right']].set_visible(False)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'validation_rate_per_plate.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — CoV of key metrics per plate — EXC / INH
if not df_conn.empty:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    plates = list(PLATE_DIRS.keys())
    x = np.arange(len(plates))
    width = 0.3

    def _cov(s):
        m = s.mean()
        return s.std(ddof=1) / m * 100 if m and not np.isnan(m) else np.nan

    metrics = [
        ('Validation rate (%)',  vr,          'rate',               'rate'),
        ('Connection count',     df_features, 'conn_n_exc',
'conn_n_inh'),
        ('AP count',             df_features, 'ap_mean_count_exc',
'ap_mean_count_inh'),
    ]

    for ax, (title, df, exc_col, inh_col) in zip(axes, metrics):
        for j, (typ, color, col) in enumerate([('EXC', QUIVER_NEURON_COLORS['Excitatory'], exc_col),
                                                ('INH', QUIVER_NEURON_COLORS['Inhibitory'], inh_col)]):
            if df is vr:
                covs = [_cov(df.loc[(df['plate'] == p) & (df['type'] == typ),
col])
                        for p in plates]
            else:
                covs = [_cov(df.loc[df['plate'] == p, col].dropna())
                        for p in plates]
            offset = (j - 0.5) * width
            ax.bar(x + offset, covs, width=width, color=color, label=typ)

        ax.set_xticks(x)
        ax.set_xticklabels(plates, rotation=30, ha='right', fontsize=8)
        ax.set_ylabel('CoV (%)')
        ax.set_title(title)
        ax.legend(frameon=False)
        ax.spines[['top', 'right']].set_visible(False)

    plt.suptitle('Within-plate variability (CoV) across plates', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'cov_metrics.png'), dpi=300,
  bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX — Compute stability metrics per FOV
if not df_conn.empty:
    PIXEL_SIZE_UM = 8.0
    VAL_TIERS = {'both', 'scs_only', 'sta_only'}

    stab_rows = []
    for R in all_fov_results:
        fov        = R['fov']
        plate      = R['_plate']
        meta       = R.get('meta', {})
        scs        = R.get('scs1')
        fov_merged = R.get('merged')

        fs    = int(meta.get('fs', 500) or 500)
        rec_s = (int(meta.get('spont_n_samples', 0) or 0) +
                int(meta.get('stim_n_samples',  0) or 0)) / fs

        if scs:
            epsp_n = sum(len(v) for v in scs.get('epsps_all', {}).values())
            ipsp_n = sum(len(v) for v in scs.get('ipsps_all', {}).values())
        else:
            epsp_n = ipsp_n = 0
        psp_rate_exc = epsp_n / rec_s if rec_s else np.nan
        psp_rate_inh = ipsp_n / rec_s if rec_s else np.nan

        rows_arr = np.array(meta.get('source_centroid_row', []), dtype=float)
        cols_arr = np.array(meta.get('source_centroid_col', []), dtype=float)
        mean_dist = np.nan
        if fov_merged is not None and not fov_merged.empty and len(rows_arr):
            val = fov_merged[fov_merged['consensus_tier'].isin(VAL_TIERS)]
            dists = []
            for _, conn in val.iterrows():
                try:
                    pre_idx  = int(str(conn['pre']).lstrip('T'))  - 1
                    post_idx = int(str(conn['post']).lstrip('T')) - 1
                    if 0 <= pre_idx < len(rows_arr) and 0 <= post_idx < len(rows_arr):
                        d = np.sqrt((rows_arr[pre_idx] - rows_arr[post_idx])**2 +
                                    (cols_arr[pre_idx] - cols_arr[post_idx])**2)
                        dists.append(d * PIXEL_SIZE_UM)
                except (ValueError, TypeError):
                    continue
            mean_dist = float(np.mean(dists)) if dists else np.nan

        stab_rows.append({
            'fov':          fov,
            'plate':        plate,
            'psp_rate_exc': psp_rate_exc,
            'psp_rate_inh': psp_rate_inh,
            'mean_dist_um': mean_dist,
        })

    df_stab = pd.DataFrame(stab_rows).merge(
        df_features[['fov', 'plate', 'neuron_n_total',
                    'ap_mean_rate_exc', 'ap_mean_rate_inh', 'conn_n_total']], on=['fov', 'plate'], how='left'
    )
    
    df_features = df_features.merge(
        df_stab[['fov', 'plate', 'mean_dist_um']],  # drop psp_rate_exc/inh
        on=['fov', 'plate'], how='left'
    )

# Cell XX — Plot stability metrics across plates
plates = list(PLATE_DIRS.keys())
if not df_conn.empty:
    width = 0.15

    panels = [
        ('neuron_n_total',   None,               'Active neurons',     'Active neurons',  0.2),
        ('conn_n_total',     None,               'Total connections', 'Connections',     0.2),
        ('mean_dist_um',     None,               'Distance (µm)', 'Distance (µm)',   0.2),
        ('ap_mean_rate_exc', 'ap_mean_rate_inh', 'Spike frequency (Hz)', 'Spike freq (Hz)', 0.4),
        ('psp_rate_exc',     'psp_rate_inh',     'PSP rate (Hz)',      'PSP rate (Hz)',   0.4),
    ]

    fig, axes = plt.subplots(1, 5, figsize=(16, 3),
                            gridspec_kw={'width_ratios': [1, 1, 1, 1.8,
1.8]})

    for ax, (exc_col, inh_col, title, ylabel, x_scale) in zip(axes, panels):
        for i, plate in enumerate(plates):
            grp = df_stab[df_stab['plate'] == plate]
            x   = i * x_scale

            if inh_col is None:
                vals = grp[exc_col].dropna()
                n    = len(vals)
                mean = vals.mean() if n else 0
                sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
                ax.bar(x, mean, yerr=sem, color='gray', width=width,
                        capsize=5, error_kw=dict(elinewidth=1.5))
                ax.scatter([x] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
                ax.text(x, 0, f'{n} FOVs', ha='center', va='bottom', fontsize=6)
            else:
                for j, (col, color, typ) in enumerate([
                    (exc_col, QUIVER_NEURON_COLORS['Excitatory'], 'EXC'), (inh_col, QUIVER_NEURON_COLORS['Inhibitory'], 'INH')
                ]):
                    vals = grp[col].dropna()
                    n    = len(vals)
                    mean = vals.mean() if n else 0
                    sem  = vals.std(ddof=1) / np.sqrt(n) if n > 1 else 0
                    xj   = x + (j - 0.5) * (width + 0.05)
                    ax.bar(xj, mean, yerr=sem, color=color, width=width,
                            capsize=5, error_kw=dict(elinewidth=1.5),
                            label=typ if i == 0 else '')
                    ax.scatter([xj] * n, vals, color='k', s=20, zorder=5, alpha=0.6)
                ax.text(x, 0, f'{n} FOVs', ha='center', va='bottom', fontsize=6)

        ax.set_xticks([i * x_scale for i in range(len(plates))])
        ax.set_xticklabels(plates, ha='center', fontsize=8)
        ax.set_ylabel(ylabel)
        ax.set_title(title)
        ax.spines[['top', 'right']].set_visible(False)
        if inh_col:
            ax.legend(frameon=False, fontsize=7)

    plt.suptitle('Network stability across plates', fontsize=11)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'network_stability.png'), dpi=300, bbox_inches='tight')
    plt.close()

In [ ]:
# Cell XX - Variance table
styled = build_variance_table(df_features, output_path=os.path.join(OUTPUT_DIR,'variance_table.csv'))

In [ ]:
# Power analysis of No Drug condition
n_results = plot_power_analysis(
      df_features,
      df_ap=df_ap_filter,
      effect_sizes=[0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0],
      output_path=os.path.join(OUTPUT_DIR, 'power_analysis.png'),
  )


# Generates information about the pipeline

In [ ]:
raise StopIteration("Stopping here intentionally")

In [ ]:
pipeline_params = pd.DataFrame([
    ('Drug condition',            'metadata CSV',               'drug1',
                'No Drug, GABAzine, CNQX'),
    ('Well ID',                   'metadata CSV',               'wellId',
                'A01, B03, F06'),
    ('Stimulation protocol',      'meta (pickle)',
'stim_step_name',             'SingleCellMaskSwitchingComb'),
    ('Recording segments',        'meta (pickle)',
'spont/stim_n_samples',       '6907 / 32320 frames'),
    ('Stimulus frames per source','meta (pickle)',
'stim_frames_per_source',     'per-neuron frame indices'),
    ('Number of sources',         'meta (pickle)',              'n_sources',
                '239'),
    ('Sampling rate',             'meta (pickle)',              'fs',
                '500 Hz'),
    ('Compatible pipelines',      'Quiver platform',            '—',
                'Excitability, Spontaneous'),
], columns=['Parameter', 'Source', 'Key', 'Example'])

styled_params = (pipeline_params.style
                .hide(axis='index')
                .set_caption('Pipeline flexibility — supported metadata parameters')
                .set_table_styles([{'selector': 'th, td', 'props':
[('font-size', '9px')]}]))
display(styled_params)

In [ ]:
edge_cases = pd.DataFrame([
    ('Near-coincident spike timing',  'Automatic',  'PSP detection window parameters filter overlapping events'),
    ('Small amplitude PSPs',          'Automatic',  'Detrending + spike-triggered averaging recovers low-SNR PSPs'),
    ('Silent/unstimulated sources',   'Automatic',  'Flagged via stim_mask_sources; excluded from connection mapping'),
    ('Failed FOVs',                   'Automatic',  'Try/except per FOV; failed FOVs logged and skipped without stopping batch'),
    ('Duplicate neurons',             'Automatic',  'find_duplicate_neurons removes redundant sources before analysis'),
    ('Missing metadata',              'Automatic',  'Pipeline runs without metadata; condition defaults to No Drug'),
], columns=['Edge Case', 'Handling', 'Detail'])

styled_edge = (edge_cases.style
                .hide(axis='index')
                .set_caption('Pipeline adaptability — edge case handling')
                .set_table_styles([{'selector': 'th, td', 'props':
[('font-size', '9px')]}]))
display(styled_edge)

In [ ]:
version_control = pd.DataFrame([
      ('Source code',          'GitHub',       'All pipeline modules under version control'),
      ('Analysis outputs',     'GitHub',       'Output filenames include pipeline version tag'),
      ('Validation split',     'Fixed seed',   'Random seed fixed before train/test split — reproducible across runs'),
      ('Notebook versioning',  'GitHub',       'Analysis notebooks committed alongside pipeline code'),
      ('Dependency tracking',  'requirements / conda env', 'Package versions pinned in environment file'), ], columns=['Component', 'Mechanism', 'Detail'])

styled_vc = (version_control.style
            .hide(axis='index')
            .set_caption('Reproducibility — version control and analysis traceability')
            .set_table_styles([{'selector': 'th, td', 'props': [('font-size',
'9px')]}]))
display(styled_vc)